In [1]:
import pandas as pd 
import yaml

config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/global80K_config.yaml"
with open(config_path, 'r') as ymlfile:
    config = yaml.safe_load(ymlfile)

metadata_path = config['files']['creating']['metadata_table']

pre_metadata_df = pd.read_csv(metadata_path, sep='\t')
pre_metadata_df
pre_metadata_df
# change datatypes:
pre_metadata_df['seq_start'] = pre_metadata_df['seq_start'].astype('Int64').astype(str)
pre_metadata_df['seq_end'] = pre_metadata_df['seq_end'].astype('Int64').astype(str)
# set NaN values
pre_metadata_df['seq_start'] = pre_metadata_df['seq_start'].replace('<NA>', 'NaN')
pre_metadata_df['seq_end'] = pre_metadata_df['seq_end'].replace('<NA>', 'NaN')

In [2]:
pre_metadata_df

,name,sequence,category,class,source,ref_seq,seq_chr,seq_start,seq_end,seq_strand,variant_class,variant_pos,SPDI,allele,info
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2181818,2182138,+,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2182410,2182738,+,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2182832,2183099,+,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2184994,2185331,+,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2188356,2188693,+,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,scrambled,element inactive control,NaN,GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,scrambled,element inactive control,NaN,GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,scrambled,element inactive control,NaN,GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MK
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,scrambled,element inactive control,NaN,GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MK


### Control regions:
- `/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/design.control_sequences.tsv`

In [3]:
region_tbl = '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/resources/design.control_regions.tsv'
control_region_tbl = pd.read_csv(region_tbl, sep='\t')
control_region_tbl

,sample,bed_file
0,GC_Cort_Chengyu,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...
1,GC_GABA_Chengyu,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...
2,GC_Glut_Chengyu,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...
3,GC_Hon,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...
4,GC_Vista,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...
5,GC_DNase_positive,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...
6,GC_DNase_negative_brain,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...
7,GC_DNase_negative_blood,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...


In [9]:
def check_region_length(bed_df, sample, expected_length=270):
    """Check if all regions in a bed file have the expected length"""

    bed_df['length'] = bed_df['end'] - bed_df['start']
    is_twoseventy = bed_df['length'] == 270
    if is_twoseventy.sum() == bed_df.shape[0]:
        print(f'{sample} has only 270bp regions')
    else:
        print(f'{sample} has regions of different lengths')

# iterate over table and read bed files
for index, row in control_region_tbl.iterrows():
    # sample:
    sample = row['sample']
    bed_file = row['bed_file']
    # read bed file
    bed_df = pd.read_csv(bed_file, sep='\t', header=None)
    bed_df.columns = ['chr', 'start', 'end', 'id', 'score', 'strand']
    # check region length:
    check_region_length(bed_df, sample, 270)
    print(bed_file)
    print(bed_df['id'].str.contains('~').sum())
    print(bed_df.shape[0])
    break
    
    

GC_Cort_Chengyu has only 270bp regions
/data/gpfs-1/users/kisa11_c/work/coding/MPRAOligoDesign/resources/controls/Cort/primary_fetal_cortical_cells-active-hg38.bed.gz
0
195


#### Numbers of controls
- check number of controls with variants
- 